### __slots__: the key idea

In [6]:
# Normally:

class A:
    pass

a = A()
a.x = 10

print(a.__dict__)

# gives:

# {'x': 10}

{'x': 10}


In [2]:
# With __slots__:

class A:
    __slots__ = ("x",)

a = A()
a.x = 10

# x is stored without a normal instance __dict__.
# So:
print(a.x)        # 10
print(a.__dict__) # AttributeError

10


AttributeError: 'A' object has no attribute '__dict__'

In [4]:
# And because only x is allowed:

a.y = 20

# → AttributeError.

AttributeError: 'A' object has no attribute 'y' and no __dict__ for setting new attributes

In [7]:
# __setattr__

In [8]:
# This is another important dunder because it lets you control every attribute assignment.

# For example:

class Person:
    def __setattr__(self, name, value):
        print("Setting:", name, value)
        object.__setattr__(self, name, value)

p = Person()

p.name = "Shri"
p.age = 22

Setting: name Shri
Setting: age 22


In [9]:

# Output:

# Setting: name Shri
# Setting: age 22

# Every time Python sees:

# p.x = value

# it can route through:

# p.__setattr__("x", value)

# The important thing is this line:

# object.__setattr__(self, name, value)

# Without it, doing self.name = value inside __setattr__ would call __setattr__ again → infinite

In [10]:
# __delattr__

In [11]:
# Just as __setattr__ intercepts:

# a.x = 10

# __delattr__ intercepts:

# del a.x

In [ ]:
# __hash__

In [13]:
# Python uses hash() for things like:

# set()
# dict()

In [14]:
class Person:
    def __init__(self, name):
        self.name = name

p = Person("Shri")

print(hash(p))

147216165058


In [15]:
# The number will be an integer representing the object's hash.

In [17]:
# But here's the interesting part:

class Person:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        return self.name == other.name

In [18]:
p = Person("Shri")

print(hash(p))

TypeError: unhashable type: 'Person'

In [ ]:
# This will raise:

# TypeError: unhashable type

# Why?

# Because once you define a custom __eq__, Python generally disables the default __hash__.

# The reason is that hashable objects must follow:
# if a == b
#     ↓
# hash(a) == hash(b)

In [19]:
# Why does __eq__ affect __hash__?

# Suppose:

# class Person:
#     def __init__(self, name):
#         self.name = name

#     def __eq__(self, other):
#         return self.name == other.name

# Now:

# p1 = Person("Shri")
# p2 = Person("Shri")

# print(p1 == p2)

# gives:

# True

# If these objects were hashable, Python requires:
# p1 == p2
#    ↓
# hash(p1) == hash(p2)

# Otherwise a set or dict could behave incorrectly.

# Therefore Python makes the class unhashable when you define __eq__ without defining a compatible __hash__.

In [20]:
# Making it hashable

# If name is the identity of the object:

class Person:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        return self.name == other.name

    def __hash__(self):
        return hash(self.name)

In [21]:
p1 = Person("Shri")
p2 = Person("Shri")

print(p1 == p2)
print(hash(p1) == hash(p2))

True
True


In [22]:
people = {p1, p2}
print(len(people))

1


In [ ]:
# because p1 and p2 are equal and have the same hash.

# Important rule

# If a == b, then hash(a) must equal hash(b).

# But the reverse is not guaranteed:

# same hash ≠ necessarily equal

# Hash collisions can happen.

In [23]:
# ⚠️ Mutable objects

# This is dangerous:

# p = Person("Shri")
# s = {p}

# p.name = "Rahul"

# If name is used in __hash__, changing it after inserting p into a set can make the set unable to find the object correctly.

# That's why hash-based identity is generally based on immutable data.